# Chapter 7: Evaluating Retrieval-Augmented Generation (RAG)
A RAG system = **retrieve** chunks from a knowledge base → **generate** an answer from them.
Each stage can fail on its own, so we **evaluate each stage separately**.

We will build a tiny real-estate assistant and walk through:
1. Chunking documents
2. Synthetic questions (easy and hard) + filtering
3. Retrieval metrics: Precision@k, Recall@k, Hit rate, MRR, NDCG@k
4. Comparing retrievers (embedding, BM25, hybrid) and tuning chunk size with a **grid search**
5. Generation metrics: **faithfulness** and **relevance**
6. Context swapping (is the model really using the documents?)
7. Diagnosing failures: retrieval problem or generation problem?

Run the cells from top to bottom. Gemini is used for both the LLM and the embeddings (through the embedded helpers).

## Run in Google Colab or locally
Upload this notebook alone and run cells from top to bottom. Setup installs missing packages;
all required sample data and helper functions are included below.
For Gemini calls, add `GEMINI_API_KEY` in Colab's key-shaped **Secrets** panel and enable notebook access,
or enter your key at the hidden prompt. Local runs use the single repository-root `.env`.
Never paste a key into a code cell. API calls use your own quota; notebook 04 needs no API key.
Students: **File → Save a copy in Drive** for your own copy and session.
Generated files live in the current session; download any results you want to keep before it ends.


In [ ]:
# Run once per fresh Colab session. Local users can run uv sync at the repo root.
import importlib.util
import subprocess
import sys

packages = {"google.genai": "google-genai", "dotenv": "python-dotenv",
            "numpy": "numpy", "pandas": "pandas", "sklearn": "scikit-learn",
            "rank_bm25": "rank-bm25"}
def available(module):
    try:
        return importlib.util.find_spec(module) is not None
    except ModuleNotFoundError:
        return False
missing = [package for module, package in packages.items() if not available(module)]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])


In [ ]:
"""Small helpers shared by all notebooks (Gemini calls + data loading)."""
import json
import os
import time
from pathlib import Path

import numpy as np
from dotenv import load_dotenv
from google import genai
from google.genai import types

# Colab uses its own Secrets; local runs find the root .env.
try:
    from google.colab import userdata
except ImportError:
    userdata = None
if userdata is None:
    for directory in (Path.cwd(), *Path.cwd().parents):
        if (directory / ".env").is_file():
            load_dotenv(directory / ".env", override=False)
            break

MODEL = os.getenv("GEMINI_MODEL", "gemini-2.5-flash")
EMBED_MODEL = os.getenv("GEMINI_EMBED_MODEL", "gemini-embedding-001")

_client = None


def client():
    """Create the Gemini client once and reuse it."""
    global _client
    if _client is None:
        key = os.getenv("GEMINI_API_KEY")
        if not key and userdata is not None:
            try:
                key = userdata.get("GEMINI_API_KEY")
            except Exception:
                pass
        if not key:
            from getpass import getpass
            key = getpass("Enter your own Gemini API key (hidden): ").strip()
        if not key:
            raise RuntimeError("GEMINI_API_KEY is missing. Add it to Colab Secrets or the root .env.")
        _client = genai.Client(api_key=key)
    return _client



def _generate(contents, system=None, temperature=0.0, as_json=False, model=None, retries=5):
    config = types.GenerateContentConfig(
        system_instruction=system,
        temperature=temperature,
        response_mime_type="application/json" if as_json else None,
    )
    for attempt in range(retries):
        try:
            resp = client().models.generate_content(model=model or MODEL, contents=contents, config=config)
            return json.loads(resp.text) if as_json else resp.text
        except Exception as e:  # rate limits (free tier) or temporary errors
            if attempt == retries - 1:
                raise
            wait = 10 * (attempt + 1)
            print(f"[retry {attempt + 1}] {type(e).__name__}: waiting {wait}s")
            time.sleep(wait)


def ask(prompt, system=None, temperature=0.0, as_json=False, model=None):
    """One prompt in, one answer out. as_json=True returns a Python dict."""
    return _generate(prompt, system, temperature, as_json, model)


def chat(history, system=None, temperature=0.0, model=None):
    """history = [{"role": "user" | "assistant", "content": "..."}, ...]"""
    contents = [
        types.Content(role="model" if m["role"] == "assistant" else "user",
                      parts=[types.Part(text=m["content"])])
        for m in history
    ]
    return _generate(contents, system, temperature, False, model)


def embed(texts):
    """Turn a list of texts into a numpy matrix (one row per text)."""
    r = client().models.embed_content(model=EMBED_MODEL, contents=texts)
    return np.array([e.values for e in r.embeddings])


def cosine(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


# All sample prompts and data are embedded in this notebook.
def load_data():
    return {}

ASSIGNMENT_RAG = {'documents': [{'id': 'borrowing', 'text': 'The City Library lets members borrow up to 10 books at a time for a period of 21 days. Books can be renewed online two times, unless another member has reserved them. A late fee of $0.25 per day is charged for each overdue book, and the fee stops growing once it reaches $10. Members who lose a book must pay its replacement cost plus a $5 processing fee. Membership is free for residents of the city, while visitors pay $15 per year for a card. DVDs can be borrowed for 7 days only.'}, {'id': 'hours_rooms', 'text': 'The main library is open Monday to Thursday from 8 AM to 8 PM, and on Friday from 8 AM to 5 PM. On Saturdays the library opens at 10 AM and closes at 4 PM, and it is closed on Sundays and public holidays. Study rooms can be reserved up to 14 days in advance for a maximum of 3 hours per day. Each study room fits up to 6 people and has a whiteboard. Group reservations need a library card from at least one member. Printing costs $0.10 per page in black and white and $0.50 per page in color. Free wifi is available in the whole building.'}, {'id': 'events', 'text': "The library runs a children's story hour every Wednesday at 4 PM in the reading corner. It is free and suitable for ages 3 to 7. The summer reading challenge starts on June 1 and ends on August 31, and children who read 12 books receive a free ticket to the science museum. Adult book clubs meet on the last Thursday of each month, and the next book is announced two weeks before. The library also offers a free computer class for seniors on Tuesday mornings at 10 AM. Registration for all events is done at the front desk or on the library website."}], 'queries': [{'query': 'How many books can I take out at once?', 'doc_id': 'borrowing', 'answer_span': '10 books', 'reference_answer': 'Up to 10 books at a time.', 'answer_keywords': ['10']}, {'query': 'What is the late fee for an overdue book?', 'doc_id': 'borrowing', 'answer_span': '$0.25 per day', 'reference_answer': '$0.25 per day per overdue book.', 'answer_keywords': ['0.25', '25 cents']}, {'query': 'How much does a library card cost if I live outside the city?', 'doc_id': 'borrowing', 'answer_span': '$15 per year', 'reference_answer': 'Visitors pay $15 per year.', 'answer_keywords': ['15']}, {'query': 'Can I visit the library on Sunday?', 'doc_id': 'hours_rooms', 'answer_span': 'closed on Sundays', 'reference_answer': 'No, the library is closed on Sundays.', 'answer_keywords': ['closed']}, {'query': 'How long can I book a study room each day?', 'doc_id': 'hours_rooms', 'answer_span': '3 hours per day', 'reference_answer': 'A maximum of 3 hours per day.', 'answer_keywords': ['3 hours', 'three hours']}, {'query': 'What is the price to print in color?', 'doc_id': 'hours_rooms', 'answer_span': '$0.50 per page in color', 'reference_answer': '$0.50 per page.', 'answer_keywords': ['0.50', '50 cents']}, {'query': 'When is story time for kids?', 'doc_id': 'events', 'answer_span': 'every Wednesday at 4 PM', 'reference_answer': 'Every Wednesday at 4 PM.', 'answer_keywords': ['wednesday']}, {'query': 'What do children get for finishing the summer reading challenge?', 'doc_id': 'events', 'answer_span': '12 books', 'reference_answer': 'A free ticket to the science museum.', 'answer_keywords': ['museum']}]}


In [ ]:
import math, re, random
import numpy as np
import pandas as pd
from rank_bm25 import BM25Okapi
from IPython.display import display

K = 3          # we look at the top-3 chunks everywhere (our knowledge base is small)
print("Model:", MODEL)

## 1. Our data: a tiny knowledge base + test questions
- `documents`: 6 short documents of a fictional agency, *Maple Grove Realty*.
- `queries`: 12 realistic user questions. For each one we store:
  - `doc_id`: the document that has the answer
  - `answer_span`: the exact words in the document that answer it
  - `reference_answer`: the correct answer (used later to check generation)

**Why store the answer text and not a chunk number?** Chunk numbers change every time we change the chunk size.
The `answer_span` stays the same, so a chunk is relevant if it **contains the answer span**.

In [ ]:
documents = [
 {"id": "property_tax", "text": "Property taxes in Maple Grove are billed twice a year, in November and in April. The average annual property tax for a three-bedroom house is $4,200. Homeowners aged 65 or older can apply for a senior discount of 15 percent through the county office. The county office is open Monday to Friday from 9 AM to 4 PM. Late payments are charged a penalty of 1.5 percent per month. Taxes for the North Hills area are slightly higher because of the new school fund, and the average there is $5,100 per year."},
 {"id": "hoa_fees", "text": "Houses in the Lakeview community belong to a homeowners association (HOA). The monthly HOA fee is $250 and covers lawn care, snow removal, and the community pool. Residents must pay the fee by the 5th of each month. The HOA board meets on the first Tuesday of every month in the clubhouse. Painting a front door or building a fence requires written approval from the board, which usually takes 14 days. Pet owners may keep up to two dogs or cats per home, and dogs must be on a leash in shared areas."},
 {"id": "viewings", "text": "Viewings at Maple Grove Realty are by appointment only. You can book a viewing online or by calling the front desk. Each viewing lasts 45 minutes. Weekend viewings are available on Saturdays from 10 AM to 2 PM, and no viewings are held on Sundays. Please bring a photo ID, because agents must verify every visitor. Buyers who want a second viewing can request one free of charge within 7 days of the first visit. Virtual tours are available for every listing and can be booked at any time."},
 {"id": "mortgage", "text": "Most buyers at Maple Grove Realty use a mortgage. Our partner bank requires a minimum down payment of 12 percent for first-time buyers and 20 percent for all other buyers. The current fixed interest rate for a 30-year mortgage is 6.4 percent. Pre-approval takes about 3 business days and is valid for 90 days. Buyers must provide two recent pay slips and six months of bank statements. Closing costs are usually between 2 and 4 percent of the purchase price, and the buyer pays them at signing."},
 {"id": "neighborhoods", "text": "North Hills is a quiet area with large houses, and most homes have three or more bedrooms. The median price in North Hills is $520,000. Lakeview is closer to the city center and has many smaller houses, with a median price of $380,000. Public transport is good in Lakeview, with a train station 10 minutes away on foot. North Hills has two primary schools and one high school, while Lakeview has one primary school. Crime rates are low in both areas. Many families choose North Hills for the schools, and young professionals choose Lakeview for the short commute."},
 {"id": "selling", "text": "Sellers at Maple Grove Realty pay a commission of 3 percent of the final sale price. The agency takes professional photos of the house for free. A typical listing stays on the market for 38 days before it sells. Sellers should repair visible damage and remove personal items before the first viewing. An energy certificate is required by law and costs about $150. The agency can arrange the certificate within 5 working days. Sellers may cancel the listing agreement with 30 days written notice."},
]

queries = [
 {"query": "What is the average yearly property tax for a three-bedroom house?", "doc_id": "property_tax", "answer_span": "$4,200", "reference_answer": "$4,200 per year."},
 {"query": "Is there a tax discount for older homeowners?", "doc_id": "property_tax", "answer_span": "15 percent", "reference_answer": "Yes, homeowners aged 65 or older can get a 15 percent senior discount."},
 {"query": "How much do I pay each month to the homeowners association?", "doc_id": "hoa_fees", "answer_span": "$250", "reference_answer": "$250 per month."},
 {"query": "Can I have a pet in Lakeview?", "doc_id": "hoa_fees", "answer_span": "two dogs or cats", "reference_answer": "Yes, up to two dogs or cats per home."},
 {"query": "How long is a house viewing?", "doc_id": "viewings", "answer_span": "45 minutes", "reference_answer": "45 minutes."},
 {"query": "Can I see a house on a Sunday?", "doc_id": "viewings", "answer_span": "no viewings are held on Sundays", "reference_answer": "No, viewings are not held on Sundays."},
 {"query": "What down payment do first-time buyers need?", "doc_id": "mortgage", "answer_span": "12 percent", "reference_answer": "At least 12 percent."},
 {"query": "What is the interest rate for a 30-year fixed loan?", "doc_id": "mortgage", "answer_span": "6.4 percent", "reference_answer": "6.4 percent."},
 {"query": "What is the typical house price in North Hills?", "doc_id": "neighborhoods", "answer_span": "$520,000", "reference_answer": "The median price is $520,000."},
 {"query": "Which area has a train station within walking distance?", "doc_id": "neighborhoods", "answer_span": "10 minutes away on foot", "reference_answer": "Lakeview."},
 {"query": "How much commission does the agency charge sellers?", "doc_id": "selling", "answer_span": "commission of 3 percent", "reference_answer": "3 percent of the final sale price."},
 {"query": "How many days of notice do I need to cancel my listing agreement?", "doc_id": "selling", "answer_span": "30 days written notice", "reference_answer": "30 days written notice."},
]

doc_text = {d["id"]: d["text"] for d in documents}

def norm(text):
    return " ".join(text.lower().split())

# Data check: every answer span must appear in its own document, and in no other document.
for q in queries:
    owners = [d["id"] for d in documents if norm(q["answer_span"]) in norm(d["text"])]
    assert owners == [q["doc_id"]], (q["query"], owners)

print(len(documents), "documents |", sum(len(d["text"].split()) for d in documents), "words |", len(queries), "queries")

## 2. Chunking: cut documents into pieces
A **chunk** is the unit the retriever can return. We use **fixed-size chunking** with **overlap**.
To keep things simple, we count **words** (not tokens).

- `size` = words per chunk
- `overlap` = words shared with the previous chunk, so text near a boundary lands in two chunks

In [ ]:
def make_chunks(docs, size, overlap):
    """Split every document into chunks of `size` words; consecutive chunks share `overlap` words."""
    assert 0 <= overlap < size
    step = size - overlap
    chunks = []
    for doc in docs:
        words = doc["text"].split()
        start = 0
        while True:
            chunks.append({"chunk_id": len(chunks), "doc_id": doc["id"],
                           "text": " ".join(words[start:start + size])})
            if start + size >= len(words):      # this chunk reached the end of the document
                break
            start += step
    return chunks

for size, overlap in [(20, 0), (20, 10), (40, 0), (40, 20), (80, 40)]:
    print(f"size={size:3} overlap={overlap:2} -> {len(make_chunks(documents, size, overlap))} chunks")

print("\nFirst 2 chunks of the first document (size=20, overlap=10): notice the shared words")
for c in make_chunks(documents[:1], 20, 10)[:2]:
    print(f"  chunk {c['chunk_id']}: {c['text']}")

## 3. Embeddings helper
We call Gemini embeddings through `embed()` and **remember** every result, so the same text is never embedded twice
(saves time and quota). Cosine similarity = dot product of unit-length vectors.

In [ ]:
_emb_cache = {}

def embed_texts(texts, batch=50):
    """Embed a list of texts with Gemini. Returns a 2D numpy array (one row per text)."""
    todo = [t for t in dict.fromkeys(texts) if t not in _emb_cache]
    for i in range(0, len(todo), batch):
        part = todo[i:i + batch]
        for text, vec in zip(part, embed(part)):
            _emb_cache[text] = np.array(vec, dtype=float)
    return np.array([_emb_cache[t] for t in texts])

def unit(v):
    v = np.asarray(v, dtype=float)
    return v / np.linalg.norm(v, axis=-1, keepdims=True)

vecs = unit(embed_texts(["tax discount for seniors", "pensioners pay less property tax", "pool opening hours"]))
print("Embedding size:", vecs.shape[1])
print("similar meaning  :", round(float(vecs[0] @ vecs[1]), 2))
print("different meaning:", round(float(vecs[0] @ vecs[2]), 2))

## 4. Synthetic questions: easy, hard, and filtered
Writing test questions by hand is slow, so we ask an LLM to write them. For each chunk the LLM picks one fact and writes a question,
so we **know the correct chunk by construction**.

- **Easy question**: made from one chunk, usually copies its wording.
- **Hard question**: we also show a *similar* chunk B and ask for a question that only chunk A answers, but that reuses words from B. This tests whether the retriever can tell similar chunks apart.
- **Filter**: another LLM rates how *realistic* each question is (1-5). We keep only good ones. (A 1-5 scale is fine here because we are only filtering data, not measuring failures.)

In [ ]:
QA_PROMPT = """You are generating synthetic QA pairs for retrieval evaluation.
Given a chunk of text, extract one specific, self-contained fact from it.
Then write a question that is directly and unambiguously answered by that fact alone.
Reply as JSON: {"fact": "...", "question": "..."}

Chunk:
"{chunk}"
"""

HARD_PROMPT = """You are generating a HARD synthetic question for retrieval evaluation.
Chunk A contains the answer. Chunk B is similar but does NOT contain the answer.
Write ONE realistic user question that is answered only by Chunk A, but that reuses words or topics from Chunk B,
so a keyword-based retriever could confuse the two chunks.
Reply as JSON: {"fact": "...", "question": "..."}

Chunk A: "{chunk_a}"

Chunk B: "{chunk_b}"
"""

FILTER_PROMPT = """You help us curate questions for evaluating a real estate information retrieval system.
Realistic examples (good):
- "What are the property taxes for 123 Main St?"  -> rating 5
- "Show me 3-bedroom houses in North Berkeley with a backyard."  -> rating 5
Unrealistic examples (bad):
- "Tell me about the color of the third brick from the left on the chimney of the house sold yesterday."  -> rating 1
- "If I combine the square footage of all houses in zip code 94704, what is the total?"  -> rating 2

Rate the realism of this question from 1 to 5.
Question: "{question}"
Reply as JSON: {"explanation": "one short sentence", "rating": 1 to 5}"""

def to_rating(value):
    m = re.search(r"[1-5]", str(value))
    return int(m.group()) if m else 0

demo_chunks = make_chunks(documents, size=40, overlap=0)       # no overlap -> exactly one correct chunk
demo_vecs = unit(embed_texts([c["text"] for c in demo_chunks]))

rows = []
for i in random.Random(7).sample(range(len(demo_chunks)), 3):    # 3 chunks keeps the demo cheap
    chunk_a = demo_chunks[i]
    sims = demo_vecs @ demo_vecs[i]
    sims[i] = -1                                                  # do not pick the chunk itself
    chunk_b = demo_chunks[int(sims.argmax())]                     # most similar other chunk

    easy = ask(QA_PROMPT.replace("{chunk}", chunk_a["text"]), as_json=True)
    hard = ask(HARD_PROMPT.replace("{chunk_a}", chunk_a["text"]).replace("{chunk_b}", chunk_b["text"]), as_json=True)

    for kind, out in [("easy", easy), ("hard", hard)]:
        question = out["question"]
        rating = to_rating(ask(FILTER_PROMPT.replace("{question}", question), as_json=True)["rating"])
        top_chunk = int((demo_vecs @ unit(embed_texts([question]))[0]).argmax())
        rows.append({"kind": kind, "question": question, "realism": rating,
                     "retriever_found_it": top_chunk == chunk_a["chunk_id"]})

synthetic = pd.DataFrame(rows)
pd.set_option("display.max_colwidth", 90)
display(synthetic)
print("Keep only realistic questions (rating >= 4):", int((synthetic["realism"] >= 4).sum()), "of", len(synthetic))

**Warning:** synthetic questions often copy the chunk's wording, which makes retrieval look easier than it is.
Always add 20-30 **real** user questions from your logs before trusting your scores.

## 5. Retrieval metrics (written from scratch)
Every metric below takes a list of **grades in ranked order**, best-ranked chunk first.
We grade each chunk for a query:

| grade | meaning |
|---|---|
| 2 | the chunk contains the answer span (relevant) |
| 1 | the chunk is from the right document but does not contain the answer (somewhat related) |
| 0 | unrelated |

Precision, Recall, Hit rate and MRR need yes/no relevance, so we **binarize**: grade >= 2 counts as relevant.
NDCG uses the full grades.

| metric | question it answers |
|---|---|
| Precision@k | what fraction of the top k are relevant? |
| Recall@k | what fraction of **all** relevant chunks are in the top k? |
| Hit rate@k | is **at least one** relevant chunk in the top k? |
| MRR | how early is the first relevant chunk? (1/rank) |
| NDCG@k | are the best chunks near the top? (1.0 = perfect order) |

In [ ]:
def precision_at_k(grades, k, thr=2):
    return sum(g >= thr for g in grades[:k]) / k

def recall_at_k(grades, total_relevant, k, thr=2):
    if total_relevant == 0:
        return 0.0                                   # the answer is not in any chunk (e.g. split by chunking)
    return sum(g >= thr for g in grades[:k]) / total_relevant

def hit_at_k(grades, k, thr=2):
    return float(any(g >= thr for g in grades[:k]))

def reciprocal_rank(grades, k, thr=2):
    for rank, g in enumerate(grades[:k], start=1):
        if g >= thr:
            return 1 / rank
    return 0.0

def dcg_at_k(grades, k):
    return sum(g / math.log2(rank + 1) for rank, g in enumerate(grades[:k], start=1))

def ndcg_at_k(grades, k):
    """Book definition: the ideal order is the same top-k chunks sorted best-first."""
    ideal = dcg_at_k(sorted(grades[:k], reverse=True), k)
    return dcg_at_k(grades, k) / ideal if ideal > 0 else 0.0

# The position discount 1 / log2(rank + 1): rank 1 is worth 1.00, rank 10 only 0.29
print("rank discount:", {r: round(1 / math.log2(r + 1), 2) for r in [1, 2, 3, 4, 5, 10]})

# Worked example: the relevant chunk (grade 2) is at rank 2, a related chunk (grade 1) at rank 4
example = [0, 2, 0, 1, 0]
print("\nGrades in ranked order:", example)
print("Precision@5 :", precision_at_k(example, 5))
print("Recall@5    :", recall_at_k(example, total_relevant=1, k=5))
print("Hit rate@5  :", hit_at_k(example, 5))
print("RR          :", reciprocal_rank(example, 5))
print("DCG@5       :", round(dcg_at_k(example, 5), 3))
print("NDCG@5      :", round(ndcg_at_k(example, 5), 3))

# Tiny self-tests
assert precision_at_k(example, 5) == 0.2
assert recall_at_k(example, 1, 5) == 1.0
assert reciprocal_rank(example, 5) == 0.5
assert round(ndcg_at_k(example, 5), 3) == 0.643
assert ndcg_at_k([2, 1, 0], 3) == 1.0                # perfect order
print("\nMetric self-tests passed.")

**Careful with NDCG:** it only checks the *order* of what was retrieved. If every retrieved chunk is only "related" (grade 1) but they are in perfect order, NDCG is 1.0 even though the answer is missing.
Always read NDCG next to Recall or Hit rate.

In [ ]:
print("Related chunks only, perfect order:")
print("  Hit rate@3:", hit_at_k([1, 1, 0], 3), "| NDCG@3:", ndcg_at_k([1, 1, 0], 3))

## 6. Three retrievers: embedding, BM25 (keywords), hybrid (RRF)
Same ideas as Chapter 2. Each retriever returns **all chunks sorted best-first** (we cut at k when scoring).

In [ ]:
def tokens(text):
    return re.findall(r"\w+", text.lower())

def build_index(chunks):
    return {"chunks": chunks,
            "vecs": unit(embed_texts([c["text"] for c in chunks])),
            "bm25": BM25Okapi([tokens(c["text"]) for c in chunks])}

def rank_embedding(index, query):
    scores = index["vecs"] @ unit(embed_texts([query]))[0]
    return [int(i) for i in np.argsort(-scores, kind="stable")]

def rank_bm25(index, query):
    scores = index["bm25"].get_scores(tokens(query))
    return [int(i) for i in np.argsort(-scores, kind="stable")]

def rrf(*rankings, k=60):
    """Reciprocal Rank Fusion: a chunk scores 1/(k+rank) in each list; add the scores."""
    scores = {}
    for ranking in rankings:
        for rank, chunk_id in enumerate(ranking, start=1):
            scores[chunk_id] = scores.get(chunk_id, 0) + 1 / (k + rank)
    return sorted(scores, key=scores.get, reverse=True)

def rank_hybrid(index, query):
    return rrf(rank_embedding(index, query), rank_bm25(index, query))

RETRIEVERS = {"embedding": rank_embedding, "bm25": rank_bm25, "hybrid (RRF)": rank_hybrid}

def grade(chunk, q):
    if norm(q["answer_span"]) in norm(chunk["text"]):
        return 2                                     # contains the answer
    return 1 if chunk["doc_id"] == q["doc_id"] else 0

def evaluate_retrieval(index, queries, rank_fn, k=K):
    """One row per query with all retrieval metrics."""
    rows = []
    for q in queries:
        all_grades = [grade(c, q) for c in index["chunks"]]          # grade of every chunk
        order = rank_fn(index, q["query"])                           # chunk ids, best first
        ranked = [all_grades[i] for i in order]                      # grades in ranked order
        rows.append({"query": q["query"],
                     "relevant_chunks": sum(g >= 2 for g in all_grades),
                     "precision": precision_at_k(ranked, k),
                     "recall": recall_at_k(ranked, sum(g >= 2 for g in all_grades), k),
                     "hit": hit_at_k(ranked, k),
                     "rr": reciprocal_rank(ranked, k),
                     "ndcg": ndcg_at_k(ranked, k),
                     "top1_chunk": order[0]})
    return pd.DataFrame(rows)

def summarize(df, k=K):
    return {f"Precision@{k}": df["precision"].mean(), f"Recall@{k}": df["recall"].mean(),
            f"Hit@{k}": df["hit"].mean(), f"MRR@{k}": df["rr"].mean(), f"NDCG@{k}": df["ndcg"].mean()}

BASE_SIZE, BASE_OVERLAP = 40, 20
base_chunks = make_chunks(documents, BASE_SIZE, BASE_OVERLAP)
base_index = build_index(base_chunks)

results = {name: evaluate_retrieval(base_index, queries, fn) for name, fn in RETRIEVERS.items()}
print(f"Chunk size {BASE_SIZE}, overlap {BASE_OVERLAP} -> {len(base_chunks)} chunks, {len(queries)} queries")
display(pd.DataFrame({name: summarize(df) for name, df in results.items()}).T.round(2))
print("Precision@3 is low on purpose: most questions have only 1-2 relevant chunks, so it cannot reach 1.0.")

### Look at the failures (this is error analysis again)
Averages hide the story. Let's read the queries where the embedding retriever did **not** find the answer in the top 3.

In [ ]:
emb = results["embedding"]
failed = emb[emb["hit"] == 0]
print(f"{len(failed)} of {len(emb)} queries missed the answer in the top {K}\n")
for _, row in failed.iterrows():
    top = base_chunks[int(row["top1_chunk"])]
    print("QUERY :", row["query"])
    print("TOP 1 :", f"[{top['doc_id']}]", top["text"][:110], "...\n")
if failed.empty:
    print("No misses with this retriever and chunking. Try the grid search below to see where chunking hurts.")

## 7. Chunking grid search
There is no best chunk size for everyone, so we **try several and measure**:
for every (size, overlap) pair we re-chunk the documents, re-index them and score the same queries.

How to read the table:
- **Hit@3** is the most practical first metric: did the answer reach the top 3?
- **Recall@3** can *drop* when overlap is added, because the same answer now sits in two chunks (two relevant chunks, and we found one). That does not mean the answer was lost, so read it next to Hit@3.
- Chunks that are too small split the answer across two chunks (`answer_not_in_any_chunk`).
- `words_sent_to_llm` = top-k chunks x chunk size. Big chunks look better partly because they cover more text, but they cost more tokens and can add noise. In a tiny corpus this advantage is exaggerated, so judge the trade-off, not just the best score.
- With only 12 queries, one query is worth 0.08, so small differences are noise.

In [ ]:
GRID = [(20, 0), (20, 10), (40, 0), (40, 20), (80, 0), (80, 40)]

rows = []
for size, overlap in GRID:
    chunks = make_chunks(documents, size, overlap)
    df = evaluate_retrieval(build_index(chunks), queries, rank_embedding)
    rows.append({"chunk_size": size, "overlap": overlap, "n_chunks": len(chunks),
                 "answer_not_in_any_chunk": int((df["relevant_chunks"] == 0).sum()),
                 "words_sent_to_llm": K * size,
                 f"Hit@{K}": df["hit"].mean(), f"Recall@{K}": df["recall"].mean(),
                 f"MRR@{K}": df["rr"].mean(), f"NDCG@{K}": df["ndcg"].mean()})

grid = pd.DataFrame(rows).round(2)
display(grid)

best = grid.sort_values([f"Hit@{K}", f"MRR@{K}", f"NDCG@{K}"], ascending=False).iloc[0]
BEST_SIZE, BEST_OVERLAP = int(best["chunk_size"]), int(best["overlap"])
print(f"Best setting (by Hit@{K}, then MRR, then NDCG): size={BEST_SIZE}, overlap={BEST_OVERLAP}")

## 8. Evaluating generation
Now the second stage. We build the full RAG answer function: retrieve top-k chunks, then ask Gemini to answer **only** from them.

In [ ]:
best_chunks = make_chunks(documents, BEST_SIZE, BEST_OVERLAP)
best_index = build_index(best_chunks)

RAG_SYSTEM = ("You are a real estate assistant. Answer the question using ONLY the documents provided. "
              "If the documents do not contain the answer, reply exactly: I don't know based on the documents. "
              "Keep the answer to one or two sentences.")

def retrieve(index, query, k=K):
    return [index["chunks"][i] for i in rank_embedding(index, query)[:k]]

def rag_answer(query, context_chunks, system=RAG_SYSTEM):
    context = "\n\n".join(c["text"] for c in context_chunks)
    return ask(f"Documents:\n{context}\n\nQuestion: {query}", system=system, temperature=0).strip()

q = queries[0]
top_chunks = retrieve(best_index, q["query"])
print("Q:", q["query"])
print("A:", rag_answer(q["query"], top_chunks))
print("Reference:", q["reference_answer"])

### Two generation metrics
- **Faithfulness**: is every claim in the answer supported by the retrieved documents? Failures: **hallucination** (made-up info), **misinterpretation** (distorted info), **omission** (an incomplete list shown as complete). Using outside knowledge counts as a failure, even if it is true.
- **Relevance**: does the answer address what the user asked? An answer can be faithful but off-topic.

We build both evaluators with the Chapter 5 tools: an **LLM judge** (binary Pass/Fail) and a simple **code check** for relevance.
Below are 7 answers with **human labels** so we can check the judges.

In [ ]:
FAITH_PROMPT = """You check whether an answer is faithful to the documents it was based on.
PASS: every claim in the answer is supported by the documents.
FAIL: the answer contains a claim that is not in the documents (hallucination), changes or distorts what the documents say (misinterpretation), or presents an incomplete list as if it were complete (omission). Using outside knowledge counts as FAIL even if it is true.

Documents:
{context}

Question: {query}
Answer: {answer}

Reply as JSON: {"reasoning": "one short sentence", "answer": "Pass" or "Fail"}"""

RELEVANCE_PROMPT = """You check whether an answer is relevant to the user's question.
PASS: the answer addresses what the user asked.
FAIL: the answer talks about something else or does not answer the question. Do not judge whether the facts are true.

Question: {query}
Answer: {answer}

Reply as JSON: {"reasoning": "one short sentence", "answer": "Pass" or "Fail"}"""

def judge_json(prompt):
    out = ask(prompt, as_json=True)
    if isinstance(out, list):                 # very rarely the model wraps the JSON in a list
        out = out[0]
    verdict = str(out.get("answer", "")).strip().capitalize()
    assert verdict in ("Pass", "Fail"), f"Unexpected judge output: {out}"
    return verdict, str(out.get("reasoning", ""))

def judge_faithfulness(context, query, answer):
    p = FAITH_PROMPT.replace("{context}", context).replace("{query}", query).replace("{answer}", answer)
    return judge_json(p)

def judge_relevance(query, answer):
    return judge_json(RELEVANCE_PROMPT.replace("{query}", query).replace("{answer}", answer))

def mentions_topic(answer, topic_words):
    """Code-based relevance check: the answer mentions at least one word of the topic."""
    return any(w in answer.lower() for w in topic_words)

labeled = [
 {"name": "good", "query": "Can I keep a pet?", "ctx": ["hoa_fees"], "topic": ["pet", "dog", "cat"],
  "answer": "Yes, you may keep up to two dogs or cats per home, and dogs must be on a leash in shared areas.",
  "human_faithful": "Pass", "human_relevant": "Pass"},
 {"name": "hallucination", "query": "What does the HOA fee cover?", "ctx": ["hoa_fees"], "topic": ["hoa", "fee", "cover"],
  "answer": "It covers lawn care, snow removal, the community pool, and free gym access.",
  "human_faithful": "Fail", "human_relevant": "Pass"},
 {"name": "misinterpretation", "query": "When must the HOA fee be paid?", "ctx": ["hoa_fees"], "topic": ["hoa", "fee", "paid", "pay"],
  "answer": "The HOA fee must be paid by the 5th of each year.",
  "human_faithful": "Fail", "human_relevant": "Pass"},
 {"name": "omission", "query": "What does the monthly HOA fee cover?", "ctx": ["hoa_fees"], "topic": ["hoa", "fee", "cover"],
  "answer": "The fee covers lawn care.",
  "human_faithful": "Fail", "human_relevant": "Pass"},
 {"name": "faithful but off-topic", "query": "What are the HOA fees?", "ctx": ["hoa_fees", "property_tax"], "topic": ["hoa", "fee"],
  "answer": "The average annual property tax for a three-bedroom house is $4,200.",
  "human_faithful": "Pass", "human_relevant": "Fail"},
 {"name": "outside knowledge", "query": "What is the interest rate for a 30-year fixed mortgage?", "ctx": ["mortgage"], "topic": ["interest", "rate", "percent"],
  "answer": "The rate is around 7 percent.",
  "human_faithful": "Fail", "human_relevant": "Pass"},
 {"name": "good (mortgage)", "query": "How long is a mortgage pre-approval valid?", "ctx": ["mortgage"], "topic": ["pre-approval", "valid", "days"],
  "answer": "Pre-approval is valid for 90 days.",
  "human_faithful": "Pass", "human_relevant": "Pass"},
]

rows = []
for ex in labeled:
    context = "\n\n".join(doc_text[d] for d in ex["ctx"])
    faithful, why_f = judge_faithfulness(context, ex["query"], ex["answer"])
    relevant, why_r = judge_relevance(ex["query"], ex["answer"])
    rows.append({"case": ex["name"],
                 "faith_human": ex["human_faithful"], "faith_judge": faithful,
                 "relev_human": ex["human_relevant"], "relev_judge": relevant,
                 "relev_code": "Pass" if mentions_topic(ex["answer"], ex["topic"]) else "Fail"})

judged = pd.DataFrame(rows)
display(judged)
print("Faithfulness judge agrees with humans:", int((judged.faith_human == judged.faith_judge).sum()), "/", len(judged))
print("Relevance judge agrees with humans   :", int((judged.relev_human == judged.relev_judge).sum()), "/", len(judged))
print("Relevance code check agrees          :", int((judged.relev_human == judged.relev_code).sum()), "/", len(judged))

Read the mismatches like in Chapter 5: if a judge is wrong, improve its definitions and test again (use a dev/test split before trusting it at scale).
The code check is cheap but crude, while the LLM judge understands meaning. Use code when the rule is objective.

## 9. Context swapping: is the model really using the documents?
Replace the retrieved documents with **unrelated** ones and ask the same question.
- If the answer **stays the same**, the model is ignoring the documents and using its own knowledge (a grounding problem).
- A good grounded model should say it does not know.

We test two system prompts. If the "no grounding instruction" version still answers with swapped documents, the fix is in the prompt.
This is a debugging tool, not a metric to run at scale. LLM outputs vary a little between runs.

In [ ]:
question = "What is the minimum down payment for first-time buyers?"
contexts = {"real documents": doc_text["mortgage"], "swapped (unrelated)": doc_text["neighborhoods"]}
systems = {"no grounding instruction": "You are a helpful real estate assistant.", "grounded prompt": RAG_SYSTEM}
IDK = ["don't know", "do not know", "not mentioned", "not provided", "does not contain", "doesn't contain", "no information"]

rows = []
for sys_name, system in systems.items():
    for ctx_name, ctx in contexts.items():
        answer = ask(f"Documents:\n{ctx}\n\nQuestion: {question}", system=system, temperature=0).strip()
        rows.append({"prompt": sys_name, "context": ctx_name,
                     "says_12_percent": "12" in answer,
                     "says_I_dont_know": any(p in answer.lower() for p in IDK),
                     "answer": answer[:110]})
display(pd.DataFrame(rows))
print("Expected: real documents -> 12 percent. Swapped documents -> 'I don't know'. A number with swapped documents = ignoring the context.")

## 10. Diagnose the whole pipeline: retrieval problem or generation problem?
A single "answer is correct" score cannot tell us who is to blame. We combine two separate measurements per query:
1. **Retrieval hit**: is the answer span inside the top-k chunks? (code check)
2. **Answer correct**: does the generated answer match the reference answer? (LLM judge, Pass/Fail)

| retrieval hit | answer correct | meaning |
|---|---|---|
| yes | yes | OK |
| yes | no | **generation problem** (fix the prompt or model) |
| no | yes | lucky / used its own knowledge (not a real success) |
| no | no | **retrieval problem** (fix chunking or retriever) |

We also run the faithfulness judge on every answer.

In [ ]:
CORRECT_PROMPT = """Compare the model's answer with the reference answer.
PASS: the model's answer states the same key fact as the reference answer.
FAIL: the answer is missing the key fact, contradicts it, or says it does not know.

Question: {query}
Reference answer: {reference}
Model answer: {answer}

Reply as JSON: {"reasoning": "one short sentence", "answer": "Pass" or "Fail"}"""

def quadrant(hit, correct):
    if hit and correct:     return "OK"
    if hit and not correct: return "generation problem"
    if correct:             return "lucky (no evidence retrieved)"
    return "retrieval problem"

N_E2E = 8                          # number of queries to run end to end (each uses 3 Gemini calls)
rows = []
for q in queries[:N_E2E]:
    top = retrieve(best_index, q["query"])
    answer = rag_answer(q["query"], top)
    hit = any(norm(q["answer_span"]) in norm(c["text"]) for c in top)
    correct, _ = judge_json(CORRECT_PROMPT.replace("{query}", q["query"]).replace("{reference}", q["reference_answer"]).replace("{answer}", answer))
    faithful, _ = judge_faithfulness("\n\n".join(c["text"] for c in top), q["query"], answer)
    rows.append({"query": q["query"][:50], "retrieval_hit": hit, "answer_correct": correct == "Pass",
                 "faithful": faithful == "Pass", "diagnosis": quadrant(hit, correct == "Pass"), "answer": answer[:70]})

e2e = pd.DataFrame(rows)
display(e2e)
print(f"Chunking used: size={BEST_SIZE}, overlap={BEST_OVERLAP}, top-{K}")
print(e2e["diagnosis"].value_counts().to_string())

## 11. Common pitfalls (checklist)
- Only measuring the **final answer**: you cannot tell if retrieval or generation is to blame. Measure each stage.
- Trusting **synthetic questions** too much: they are often easier than real ones. Add real queries.
- Using the **wrong metric**: one fact needed -> Hit rate / MRR. Many chunks needed -> Recall / NDCG.
- Ignoring **chunking**: it can change retrieval scores a lot with the same retriever.
- Judging generation **without checking grounding**: fluent is not the same as faithful.
- Not separating "retrieval worked, answer wrong" from "retrieval failed, answer lucky".

## Workflow recap
1. Build a dataset of queries with labeled answer chunks.
2. Measure retrieval (Hit/Recall for coverage, MRR/NDCG for ranking).
3. Separately measure generation (faithfulness and relevance).
4. Diagnose: retrieval error or generation error?
5. Treat chunking as a variable and tune it with a grid search.